# Gaze-driven VLM input on Gemma-3-4B: Full gate (when) + foveated tokens (where)

The same pipeline as `colab_vlm_egoeverything.ipynb` (Qwen2.5-VL-3B), on Google's
**Gemma-3-4B-it**:

```
gaze + IMU -> Full gate (eye + head model) -> policy D: which frames are sent
sent frame (896 px) -> SigLIP (Gemma's own encoder, 64 x 64 patches)
                    -> foveated pooling: 4 x 4 patches (56 px) per token at the gaze,
                       8 x 8 (112 px) and 16 x 16 (224 px) per token further out
                    -> Gemma's own norm + projection -> Gemma-3-4B LLM   (no training)
```

Gemma averages each 4 x 4 block of patch features into one token (256 per frame); that step
has no weights, so foveation simply averages larger blocks in the periphery. **Fovea tokens
are exactly Gemma's own tokens** (nothing drifts), and the encoder still sees the full frame;
the saving is in LLM tokens. Gemma's 16 x 16 token grid on 896 px has the same geometry as
Qwen's on 448 px, so the **same ring settings give the same layouts and token counts** as the
Qwen run, and the two models are directly comparable.

| condition (all on policy D's frames) | tokens per frame |
|---|---|
| Gemma stock (4 x 4 pooling) | 256 |
| **fovea at gaze**, 4 budgets | ~49 / 98 / 151 / 199 |
| uniform pooling to the same tokens (s x s), per budget | same |
| fovea at image centre, middle budget | ~136 |

**Gemma is gated on Hugging Face:** accept the license at
https://huggingface.co/google/gemma-3-4b-it with your account, and add a read token as the
Colab secret `HF_TOKEN` (or paste it when section 1 asks).

## Run order

1. Clone, mount, install, Hugging Face login  2. AEA links  3. Questions  4. Full gate + policy D
5. Frames and gaze  6. Gemma: unit tests, parity with Gemma's own pipeline  7. Score  8. Results

## 1. Clone, mount, install, Hugging Face login

In [ ]:
import os, sys, json, subprocess

REPO, BRANCH = '/content/GazeVLM', 'eyehead-gate'     # 'main' once merged
URL = 'https://github.com/shubhamOjha1000/GazeVLM-HWSW-Codesign.git'
if os.path.exists(os.path.join(REPO, '.git')):
    subprocess.run(['git', '-C', REPO, 'fetch', '-q', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO, 'checkout', '-q', '-B', BRANCH, f'origin/{BRANCH}'], check=True)
else:
    subprocess.run(['git', 'clone', '-q', '-b', BRANCH, URL, REPO], check=True)
os.chdir(REPO)
sys.path.insert(0, REPO)
print(subprocess.run(['git', 'log', '--oneline', '-1'], capture_output=True, text=True).stdout)
assert os.path.exists('vlmeval/gemma_fovea.py'), f'vlmeval/gemma_fovea.py is not on branch {BRANCH!r}'

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'huggingface_hub', 'pytest'], check=True)

import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/GazeVLM'
FOLDS     = os.path.join(DRIVE_DIR, 'folds')
RUNS      = os.path.join(DRIVE_DIR, 'runs')
VLM_DIR   = os.path.join(DRIVE_DIR, 'vlm', 'egoeverything')
FRAMES_TAR = os.path.join(VLM_DIR, 'frames_tar')          # one .tar per video and size, reused later
LOCAL_HI = '/content/frames_1344'
os.makedirs(VLM_DIR, exist_ok=True)

CKPT_FULL = os.path.join(RUNS, 'eyehead')                  # the Full model's 5 fold checkpoints
missing = [i for i in range(1, 6) if not os.path.exists(os.path.join(CKPT_FULL, f'fold{i}.pt'))]
assert not missing, f'missing fold checkpoints {missing} in {CKPT_FULL} -- run the eyehead notebook first'
print('Full model checkpoints: 5 folds found in', CKPT_FULL)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')

In [ ]:
# Gemma is gated: log in with a token from an account that accepted the license
from huggingface_hub import login, model_info
MODEL_ID = 'google/gemma-3-4b-it'
token = None
try:
    from google.colab import userdata
    token = userdata.get('HF_TOKEN')
except Exception as ex:
    print('no HF_TOKEN Colab secret available:', type(ex).__name__)
if not token:                                # login() would show a non-blocking widget; ask instead
    from getpass import getpass
    token = getpass('Hugging Face read token (account that accepted the Gemma license): ')
login(token=token)
try:
    model_info(MODEL_ID)
    print('access to', MODEL_ID, 'OK')
except Exception as ex:
    raise RuntimeError(f'no access to {MODEL_ID}: open https://huggingface.co/{MODEL_ID}, accept the '
                       f'license with the same account, then rerun this cell') from ex

## 2. The AEA download-links JSON

The signed links expire after ~14 days. This cell checks the copy on Drive:

- **missing or expired** → it asks you to upload the fresh `aea_download_urls.json`, checks
  its links work, and **replaces the older copies** on Drive (`MyDrive/GazeVLM/` and, if
  present, `MyDrive/aea/`, so every notebook picks up the fresh one)
- **still valid** → nothing to do

Set `FORCE_UPLOAD = True` to replace it anyway.

In [ ]:
from vlmeval.frames import links_fresh, install_links

FORCE_UPLOAD = False
LINKS = os.path.join(DRIVE_DIR, 'aea_download_urls.json')
OLD_COPIES = [LINKS] + [p for p in ['/content/drive/MyDrive/aea/aea_download_urls.json']
                        if os.path.exists(p)]

ok, msg = links_fresh(LINKS) if os.path.exists(LINKS) else (False, 'not on Drive')
print('Drive copy:', msg)
if FORCE_UPLOAD or not ok:
    from google.colab import files
    print('upload the fresh aea_download_urls.json ...')
    up = files.upload()
    name = list(up.keys())[0]
    ok, msg = links_fresh(name)
    print('uploaded file:', msg)
    assert ok, 'the uploaded JSON does not work either -- download a fresh one from projectaria.com/datasets/aea'
    install_links(name, OLD_COPIES)
    os.remove(name)                      # the copies live on Drive; keep the repo clean
    for p in OLD_COPIES:
        print('   replaced', p)
links = json.load(open(LINKS))
print(f"using {LINKS}: {len(links['sequences'])} sequences")
print('(the eyehead notebook\'s IMU calibration, section 3, can now download its VRS too)')

## 3. EgoEverything questions on the gate's videos

In [ ]:
from vlmeval import egoev

qs_all = egoev.load_questions()
aea = qs_all[qs_all.video.str.startswith('loc')]
clips_all, test = egoev.gate_clips(FOLDS, test_only=False)
gate_videos = {c['sequence'] for c in clips_all}
test_videos = {c['sequence'] for c in clips_all if c['is_test']}

# DEMO = True: only the 4 locked TEST videos (204 questions), to check that everything works.
# DEMO = False: all 12 gate videos with EgoEverything questions (834 questions).
DEMO = False
qs = aea[aea.video.isin(gate_videos)].reset_index(drop=True)
if DEMO:
    qs = qs[qs.video.isin(test_videos)].reset_index(drop=True)
clips = [c for c in clips_all if c['sequence'] in set(qs.video)]
videos = [c['sequence'] for c in clips]
n_frames = {c['sequence']: len(c['frame_no']) for c in clips}
print(f'EgoEverything: {len(qs_all):,} questions, {len(aea):,} on AEA ({aea.video.nunique()} videos)')
print(f'on the gate videos: {len(qs):,} questions on {len(videos)} videos '
      f'({qs.video.isin(test_videos).sum()} questions on {len(set(videos) & test_videos)} test videos)\n')

per = qs.groupby('video').agg(questions=('qid', 'size'), t0_max=('t0', 'max'),
                              median_dist2gaze=('dist2gaze', 'median'))
per['video_s'] = [n_frames[v] for v in per.index]
per['split'] = ['test' if v in test_videos else 'train (held out by its fold)' for v in per.index]
display(per.sort_values('questions', ascending=False).round(1))
print('answer letters:', qs.answer.value_counts().sort_index().to_dict())

## 4. Full gate + policy D: which frames are sent

Each training video is scored by the fold model that held it out; test videos by the 5-fold
ensemble (cached on Drive). The held-out 4-class accuracy should be close to the
cross-validation numbers (~0.6), not higher, which would mean a video leaked into its own
training. Policy D then sends a frame when the view settles after motion, and on each
REFIXATION.

In [ ]:
from eyehead.oof import oof_predictions

OOF = os.path.join(VLM_DIR, 'oof_preds_full.csv')
if os.path.exists(OOF):
    oof = pd.read_csv(OOF)
    print('loaded', OOF)
else:
    oof = oof_predictions(FOLDS, CKPT_FULL)
    oof.to_csv(OOF, index=False)
    print('wrote', OOF)

o12 = oof[oof.sequence.isin(videos)]
print(f"held-out 4-class accuracy on these {len(videos)} videos: {(o12.pred == o12.quad).mean():.3f}")

labels = {'full': egoev.video_labels(clips, oof)}
# Temporal policy: 'D' = send when the view settles + on each REFIXATION;
# 'D+P' = D + the first frame of every PURSUIT run; 'D+P all' = D + every PURSUIT frame.
POLICY = 'D+P'
POLICY_TAG = {'D': 'D', 'D+P': 'DP', 'D+P all': 'DPall'}[POLICY]   # separate result files per policy
sel = {'D: full': egoev.select_conditions(clips, labels, main='full')[f'{POLICY}: full']}
D = sel['D: full']
per = pd.DataFrame({'video_s': n_frames, 'frames sent': {v: len(D[v]) for v in videos}})
per['one frame every (s)'] = (per.video_s / per['frames sent']).round(1)
display(per)
print(f"policy {POLICY} sends {60 * per['frames sent'].sum() / per.video_s.sum():.1f} frames/min "
      f"({per['frames sent'].mean():.1f} per video)")

## 5. Frames and gaze

The 1344 px frames cached on Drive by the Qwen notebook are reused (each is resized to 896 px
for Gemma); videos not cached yet are downloaded once. One gaze point per sent frame: the
densest fixation cluster in the second before it (`vlmeval.spatial.gaze_fixation`).

In [ ]:
from vlmeval.frames import ensure_frames, frame_path
from vlmeval import qwen_fovea as qf, gemma_fovea as gf, spatial as sp

metas = {}
for v in videos:
    metas[v] = ensure_frames(v, links, LOCAL_HI, FRAMES_TAR, size=qf.IMG)
    m = metas[v]
    flag = 'OK' if m['n'] >= n_frames[v] else f'!! only {m["n"]} frames, labels need {n_frames[v]}'
    print(f"   {v:<26} {m['n']:4d} frames  {flag}")

def frame_time(v, k):
    return metas[v]['ts'][k] if k < metas[v]['n'] else k

tracks, n_drop = sp.gaze_tracks(FOLDS, videos, frame_time)
frames_of = {}
for v in videos:
    m = metas[v]
    frames_of[v] = []
    for f in D[v]:
        if f >= m['n']:
            continue
        t = float(m['ts'][f])
        gz_ = sp.gaze_fixation(tracks[v], t) if v in tracks else None
        frames_of[v].append(dict(key=(v, int(f)), path=frame_path(m, f), t=t,
                                 gaze=tuple(map(float, gz_)) if gz_ is not None else None))
items = [dict(qid=int(q.qid), video=q.video, question=q.question, options=list(q.options),
              answer=q.answer, frames=frames_of[q.video])
         for q in qs.sort_values(['video', 'qid']).itertuples()]
nf = np.array([len(it['frames']) for it in items])
print(f"{len(items)} questions on {len(videos)} videos; frames per question: mean {nf.mean():.1f}, max {nf.max()}; "
      f"gaze missing on {sum(fr['gaze'] is None for v in videos for fr in frames_of[v])} of "
      f"{sum(len(frames_of[v]) for v in videos)} sent frames")

## 6. Gemma-3-4B: unit tests and parity with Gemma's own pipeline

| check | pass |
|---|---|
| unit tests (CPU, tiny projector from transformers' own `Gemma3MultiModalProjector`): cells cover the 64 x 64 grid once, token counts = Qwen 448 px layouts, stock 4 x 4 pooling = Gemma's, **projector path = stock projector**, **fovea tokens = Gemma's own tokens exactly**, condition layouts, prompt expansion | all pass |
| real model, one frame, stock 4 x 4 layout through our path vs Gemma's processor + model: pixels, vision features, token ids, answer probabilities | identical up to bf16 noise, same answer |

In [ ]:
r = subprocess.run([sys.executable, '-m', 'pytest', 'tests/test_gemma_fovea.py', 'tests/test_gaze_fixation.py',
                    '-q', '-rs'], capture_output=True, text=True, cwd=REPO)
import transformers
print('transformers', transformers.__version__, '| torch', torch.__version__)
print(r.stdout[-4000:], r.stderr[-2000:])
assert r.returncode == 0, 'unit tests failed: fix before using the GPU'

from vlmeval import qwen_mcq                      # same prompt (frame times, question, options)
model, proc, letter_ids = gf.load_model(MODEL_ID)
print(MODEL_ID, f'{torch.cuda.memory_allocated()/1e9:.1f} GB on GPU')

q0 = qs.iloc[0]
m0 = metas[q0.video]
k0 = int(np.argmin(np.abs(np.array(m0['ts']) - q0.t0)))
img0 = qf.load_frame01(frame_path(m0, k0))
rep = gf.parity_report(model, proc, letter_ids, img0, q0.question, q0.options, qwen_mcq.build_messages)
print("parity, Gemma's own 4 x 4 layout through our path vs Gemma's processor + model:")
for k_, v_ in rep.items():
    print(f'   {k_:<22} {v_}')
ok = (rep['pixel_max_abs_diff'] < 2e-2 and rep['vision_cos_min'] > 0.999 and rep['tokens_ids_equal']
      and rep['probs_max_abs_diff'] < 0.02 and rep['same_answer'])
assert ok, 'parity failed: do not trust the foveated results'
print('parity OK')

# ring settings: the same (R, a) as the Qwen run -> the same layouts and token counts
PICKS = [dict(R=70, a=160, label=48), dict(R=180, a=40, label=96),
         dict(R=180, a=160, label=150), dict(R=240, a=160, label=205)]
gz_D = [fr['gaze'] or (0.5, 0.5) for v in videos for fr in frames_of[v]]
budget = pd.DataFrame({p['label']: {'R': p['R'], 'a': p['a'], 'tokens per frame (mean over D frames)':
                       np.mean([len(gf.fovea_cells(*g_, a=p['a'], R_deg=p['R'])) for g_ in gz_D])} for p in PICKS}).T
budget['saving vs 256'] = (100 * (1 - budget.iloc[:, 2] / 256)).round(1).astype(str) + '%'
print('\ntoken budgets (Gemma stock = 256 per frame):'); display(budget.round(1))

COLS = {4: 'white', 8: 'yellow', 16: 'deepskyblue'}
v_ = videos[0]
fr_ = frames_of[v_][len(frames_of[v_]) // 2]
im_ = qf.resize01(qf.load_frame01(fr_['path']), gf.G_IMG).permute(1, 2, 0).numpy()
g_ = fr_['gaze'] or (0.5, 0.5)
fig, axs = plt.subplots(1, len(PICKS) + 1, figsize=(3.6 * (len(PICKS) + 1), 3.9))
axs[0].imshow(im_); axs[0].set_title('Gemma stock: 256 tokens', fontsize=8); axs[0].axis('off')
for a_, p in zip(axs[1:], PICKS):
    cells = gf.fovea_cells(*g_, a=p['a'], R_deg=p['R'])
    a_.imshow(im_)
    for y, x, s in cells:
        a_.add_patch(mpatches.Rectangle((x * 14, y * 14), s * 14, s * 14, fill=False, ec=COLS[s], lw=0.6))
    a_.plot(g_[0] * gf.G_IMG, g_[1] * gf.G_IMG, 'r+', ms=12, mew=2)
    a_.set_title(f"{len(cells)} tokens (R={p['R']}, a={p['a']})", fontsize=8); a_.axis('off')
plt.suptitle('one token per box: white 56 px (= Gemma own token), yellow 112 px, blue 224 px; + = gaze', fontsize=9)
plt.tight_layout(); plt.show()

## 7. Score: policy D's frames, Gemma stock vs foveated vs uniform pooling

Every question gets all the frames policy D sent on its video. SigLIP runs once per frame
and video; each condition only changes the pooling. Answer = argmax of the raw next-token
logits over A-E. Resumable. About 0.3-0.5 s per question and condition on an A100.

In [ ]:
def fov(p, src='gaze'):
    return f"D + Gemma fovea {src}, ~{p['label']} tok (R={p['R']}, a={p['a']})"

STOCK = 'D + Gemma stock (256 tokens)'
CONDS = {STOCK: ('stock', None)}
for p in PICKS:
    CONDS[fov(p)] = ('fovea', dict(a=p['a'], R=p['R'], gaze='gaze'))
    CONDS[f"D + Gemma uniform pooled, same tokens ~{p['label']}"] = ('uniform_match', fov(p))
p_mid = PICKS[len(PICKS) // 2]
CONDS[fov(p_mid, 'centre')] = ('fovea', dict(a=p_mid['a'], R=p_mid['R'], gaze='centre'))
print(f'{len(items)} questions x {len(CONDS)} conditions:', *CONDS, sep='\n   ')

OUT = os.path.join(VLM_DIR, f'gemma-3-4b-it_{POLICY_TAG}_fovea.jsonl')
gf.run_questions(model, proc, letter_ids, items, CONDS, OUT, qwen_mcq.build_messages)

## 8. Results

- accuracy per condition (95% interval from resampling videos), tokens per frame, saving vs 256
- paired tests at each budget: fovea vs uniform pooling at the same tokens, fovea vs stock
- **one pre-specified test across budgets**: per question, the mean over the 4 fovea budgets
  minus the mean over the 4 matched uniform budgets (no multiple-comparison problem)
- accuracy vs tokens plot, and the Qwen2.5-VL-3B run side by side when its results are on Drive

In [ ]:
res = egoev.load_results(OUT)
res = res[res.video.isin(videos) & res.condition.isin(CONDS)]
bad = res[res['error'].notna()] if 'error' in res else res.iloc[:0]
if len(bad):
    print(f'{len(bad)} rows not scored, left out:'); display(bad.groupby(['condition', 'error']).size())
res = res[res['error'].isna()].copy()
res['correct'] = res['correct'].astype(int)

acc = egoev.accuracy_table(res)
acc['tokens / frame'] = res.groupby('condition').n_image_tokens.mean() / acc['frames_per_q']
acc['saving vs 256'] = 100 * (1 - acc['tokens / frame'] / 256)
acc = acc.loc[[c for c in CONDS if c in acc.index]]
print(f'Gemma-3-4B, {res.qid.nunique()} questions on {res.video.nunique()} videos, policy D frames:')
display(acc[['n', 'accuracy', 'ci_low', 'ci_high', 'tokens / frame', 'saving vs 256']].round(1))

UNI = lambda p: f"D + Gemma uniform pooled, same tokens ~{p['label']}"
rows = []
for p in PICKS:
    for a_, b_ in ((fov(p), UNI(p)), (fov(p), STOCK)):
        if {a_, b_} <= set(res.condition):
            rows.append(dict(budget=p['label'], **egoev.paired_test(res, a_, b_)))
if {fov(p_mid), fov(p_mid, 'centre')} <= set(res.condition):
    rows.append(dict(budget=p_mid['label'], **egoev.paired_test(res, fov(p_mid), fov(p_mid, 'centre'))))
pt = pd.DataFrame(rows)
print('paired on the same questions (diff = a - b, accuracy points; CI from resampling videos):')
display(pt[['budget', 'a', 'b', 'diff', 'ci_low', 'ci_high', 'a_only_right', 'b_only_right', 'p_mcnemar']].round(3))

# one pre-specified test across the 4 budgets
w = res.pivot_table(index=['qid', 'video'], columns='condition', values='correct')
f_cols, u_cols = [fov(p) for p in PICKS], [UNI(p) for p in PICKS]
if set(f_cols + u_cols) <= set(w.columns):
    d = pd.DataFrame({'video': w.index.get_level_values('video'),
                      'diff': 100 * (w[f_cols].mean(1) - w[u_cols].mean(1)).to_numpy()})
    lo, hi = egoev.video_bootstrap(d, lambda t: t['diff'].mean())
    pos, neg = int((d['diff'] > 0).sum()), int((d['diff'] < 0).sum())
    try:
        from scipy.stats import binomtest
        p_sign = binomtest(pos, pos + neg, 0.5).pvalue
    except ImportError:
        p_sign = float('nan')
    print(f"\nACROSS BUDGETS: fovea - uniform pooling = {d['diff'].mean():+.2f} points, 95% CI [{lo:+.2f}, {hi:+.2f}] "
          f"(videos resampled); questions better / worse with fovea: {pos} / {neg}, sign test p = {p_sign:.3f}")

fig, ax = plt.subplots(figsize=(7, 4.4))
def curve(names, style, label):
    d_ = acc.loc[[n for n in names if n in acc.index]].sort_values('tokens / frame')
    ax.errorbar(d_['tokens / frame'], d_.accuracy, yerr=[d_.accuracy - d_.ci_low, d_.ci_high - d_.accuracy],
                fmt=style, capsize=3, label=label)
curve(f_cols, 'o-', 'Gemma: fovea at gaze')
curve(u_cols, 's--', 'Gemma: uniform pooling, same tokens')
curve([STOCK], 'k*', 'Gemma stock (256)')
curve([fov(p_mid, 'centre')], 'x', 'Gemma: fovea at image centre')

QWEN_OUT = os.path.join(VLM_DIR, f'Qwen2.5-VL-3B-Instruct_{POLICY_TAG}_fovea.jsonl')
if os.path.exists(QWEN_OUT):
    qr = egoev.load_results(QWEN_OUT)
    qr = qr[qr.video.isin(videos) & qr['error'].isna()].copy()
    qr['correct'] = qr['correct'].astype(int)
    qn = lambda p: f"D + fovea@448 gaze, ~{p['label']} tok (R={p['R']:g}, a={p['a']:g})"
    qu = lambda p: f"D + uniform, same tokens as fovea@448 ~{p['label']}"
    side = []
    for p in PICKS:
        row = {'budget': p['label']}
        for lab, name, src in (('Gemma fovea', fov(p), res), ('Gemma uniform', UNI(p), res),
                               ('Qwen fovea', qn(p), qr), ('Qwen uniform', qu(p), qr)):
            g_ = src[src.condition == name]
            row[lab] = 100 * g_.correct.mean() if len(g_) else np.nan
        side.append(row)
    side.append({'budget': 256, 'Gemma uniform': 100 * res[res.condition == STOCK].correct.mean(),
                 'Qwen uniform': 100 * qr[qr.condition == 'D + uniform 448 (256 tokens)'].correct.mean()})
    print('\nGemma-3-4B vs Qwen2.5-VL-3B, same frames, same layouts (accuracy %):')
    display(pd.DataFrame(side).set_index('budget').round(1))
ax.set(xlabel='visual tokens per frame', ylabel='accuracy (%)',
       title=f'Gemma-3-4B, policy D frames, {res.qid.nunique()} questions (95% CI over videos)')
ax.grid(alpha=.3); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

---

## Reading it

| look at | meaning |
|---|---|
| section 6 parity | must pass, or nothing after it is trustworthy |
| **across-budgets test** | the single main result: does foveated pooling beat uniform pooling at equal tokens? |
| fovea vs stock 256 | the token saving: how few tokens keep Gemma's accuracy |
| fovea at gaze vs centre | is it the gaze, or the centre bias of head-mounted video? |
| Gemma vs Qwen table | does the same foveation help a second, different VLM? |

## Caveats

- Saving is in LLM tokens; SigLIP still encodes the full 896 px frame (pre-encoder foveation,
  as done for Qwen, would also cut encoder compute).
- EgoEverything barely needs resolution (see the Qwen notebook); the across-budgets test is
  the honest summary.